# Variant-aware scoring tutorial

This notebook is a step-by-step, executable walkthrough of variant_aware.py.
It focuses on variant-aware scoring for a single example variant dataset (FASTA with variant-annotated headers).

```{note}
This example uses a single variant dataset for illustration purposes. To replicate manuscript-scale GWAS analyses and visualisation workflows, please refer to `docs/tutorials/notebooks/gwas/gwas.ipynb`.
```

## Inputs and outputs at a glance

**Input**

A FASTA file where each record contains:
  - a header encoding the variant position and REF/ALT alleles
  - a single-line sequence window (the sequence line must be one line per record)

**Output**

A plain-text file where each line contains:
  - the header (without >)
  - a Prediction_score:<float> produced by BRIDGE



## 0) Ensure we are running from the repository root (must contain `utils/`)

This step checks that you're running the notebook from the **BRIDGE repository's root folder**, which must contain the `utils/` folder. 
If you're not in the correct folder, the script will raise an error. This is important to ensure that the notebook can access the necessary utilities and modules.


In [1]:
from pathlib import Path
import os, sys

cwd = Path.cwd()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "utils").exists():
        repo_root = p
        break

if repo_root is None:
    raise RuntimeError(
        f"Cannot find repo root (no utils/ found) from {cwd}. "
        "Open this notebook from the BRIDGE repository root."
    )

os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print("Repo root:", repo_root)


Repo root: /fs1/private/user/wangyubo/code/BRIDGE



## 1) Imports

This section imports all the necessary libraries and project-specific utilities.

In [ ]:
from __future__ import annotations

import argparse
import logging
import os
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel

# ---------------------------------------------------------------------------
# Third-party / project-specific utilities
# ---------------------------------------------------------------------------
from utils.BRIDGE import BRIDGE
from utils.gen_transformer_embedding import build_Transformer_embeddings
from utils.train_loop import validate_without_sigmoid
from utils.utils import RBPInferDataset
from utils.FeatureEncoding import dealwithdata2


/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



## 2) Constants and FASTA I/O helpers

Here, we define constants like `COMPLEMENT` (for nucleotide base pair matching) and helper functions for reading FASTA files.

- `read_fasta`: Parses the FASTA file, which contains sequence data and headers, and returns two lists—one for sequence names and another for the corresponding sequences.
- `open_output`: Ensures that the output directory exists before saving results.

These functions are essential for handling the input and output data formats throughout the process.


In [3]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
COMPLEMENT: Dict[str, str] = {"A": "T", "T": "A", "C": "G", "G": "C"}

# ---------------------------------------------------------------------------
# I/O helpers
# ---------------------------------------------------------------------------
def read_fasta(fasta_path: Path) -> Tuple[List[str], List[str]]:
    """Read a *single-line FASTA* file.

    Parameters
    ----------
    fasta_path : Path
        Path to a FASTA file where each record is:
        - one header line starting with '>'
        - one sequence line (no line wrapping)

    Returns
    -------
    names : List[str]
        Header lines (including leading '>'), one per record.
    seqs : List[str]
        Uppercased sequences, one per record.

    Notes
    -----
    - This reader does not support multi-line sequences.
    - It preserves header text verbatim (minus the trailing newline).
    """
    names, seqs = [], []
    with fasta_path.open() as handle:
        for line in handle:
            if line.startswith(">"):
                names.append(line.rstrip("\n"))
            else:
                seqs.append(line.rstrip("\n").upper())
    return names, seqs


def open_output(out_path: os.PathLike | str) -> Path:
    """Ensure output directory exists and return a Path handle.

    This helper only ensures `out_path.parent` exists. The file is opened by the caller.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    return out_path


## 3) Variant utilities: header parsing + strand-aware substitution

This section contains the variant-aware parsing and sequence-edit helpers.

In [4]:
# ---------------------------------------------------------------------------
# Variant utilities
# ---------------------------------------------------------------------------
def parse_variant_block(fasta_header: str) -> Tuple[int, str, str, str, int]:
    """Parse a FASTA header and extract variant coordinates.

    The script relies on *positional tokens* inside the header, so documenting
    the expected format is essential for reproducibility.

    Expected header (example)
    -------------------------
    >variant_1 chr1:27891903-27892003(-)[...]{NA} 27891953:T>A ...

    Token usage in the current implementation
    -----------------------------------------
    fields = fasta_header.lstrip('>').split()

    - fields[1] is the region token like: chr1:27891903-27892003(-)[...]
      We parse:
        * strand    : text between '(' and ')', e.g. '+' or '-'
        * seq_start : the window start coordinate, the first number after ':'

    - fields[-2] is the variant token like: 27891953:T>A
      We parse:
        * variant_pos : the genomic position (integer)
        * ref_base    : REF base (string)
        * alt_base    : ALT base (string)

    Returns
    -------
    variant_pos : int
        Genomic coordinate of the variant.
    ref_base : str
        Reference allele base (A/C/G/T).
    alt_base : str
        Alternate allele base (A/C/G/T).
    strand : str
        '+' or '-' parsed from the region token.
    seq_start : int
        Genomic coordinate of the window start (used to compute 0-based index into the sequence).

    Raises
    ------
    ValueError
        If the header does not contain enough tokens to parse with the above rules.
    """
    fields = fasta_header.lstrip(">").split()
    if len(fields) < 3:
        raise ValueError("Unexpected FASTA header format")

    # chr region
    region = fields[1]                             # chr1:27891903-27892003(-)[...]
    strand = region.split("(")[1].split(")")[0]    # + / -
    seq_start = int(region.split(":")[1].split("-")[0])

    # variant
    var_info = fields[-2]                          # 27891953:T>A
    variant_pos = int(var_info.split(":")[0])
    ref_base, alt_base = var_info.split(":")[1].split(">")

    return variant_pos, ref_base, alt_base, strand, seq_start


def apply_complement(base: str) -> str:
    """Return Watson-Crick complement for A/T/C/G; otherwise return `base` unchanged."""
    return COMPLEMENT.get(base, base)


def substitute_base(seq: str, pos0: int, alt: str) -> str:
    """Return a new sequence where `seq[pos0]` is replaced by `alt`.

    Parameters
    ----------
    seq : str
        Input sequence (window).
    pos0 : int
        0-based index *into the window*.
    alt : str
        Alternate allele to write at `pos0`.

    Notes
    -----
    - If `seq[pos0]` already equals `alt`, we return the original string.
    """
    if seq[pos0] == alt:
        return seq  # already substituted
    seq_list = list(seq)
    seq_list[pos0] = alt
    return "".join(seq_list)



## 4) ModelHub (checkpoint caching)

The `ModelHub` class is responsible for loading and caching the BERT tokenizer and model, as well as loading checkpoint files for BRIDGE models.

- **Tokenizer and Model**: We load a pre-trained BERT model for embedding the sequences.
- **Caching**: To avoid redundant model loading, the `ModelHub` caches models and tokenizers in memory, which speeds up subsequent access during inference.
- **Loading BRIDGE model**: It retrieves the BRIDGE model checkpoint based on the provided filename stem. If the checkpoint is already cached, it is directly used; otherwise, it is loaded from disk.

This caching mechanism ensures that we avoid loading models multiple times during the process.
    

In [5]:
# ---------------------------------------------------------------------------
# Model loaders (with caching)
# ---------------------------------------------------------------------------
class ModelHub:
    """Caches heavy models & tokenizers to avoid redundant I/O.
    
    The tokenizer/transformer are loaded once and held for reuse. BRIDGE checkpoints
    are cached by filename stem to avoid repeated disk loads in long FASTA batches.
    """

    def __init__(self, transformer_path: Path, device: torch.device) -> None:
        """Initialize the hub.

        Parameters
        ----------
        transformer_path : Path
            Path to a directory compatible with `BertTokenizer.from_pretrained`
            and `BertModel.from_pretrained`.
        device : torch.device
            CPU or CUDA device for inference.
        """
        self.device = device
        self.tokenizer = BertTokenizer.from_pretrained(transformer_path, do_lower_case=False)
        self.transformer = (
            BertModel.from_pretrained(transformer_path).to(device).eval()
        )
        self.bridge_cache: Dict[str, BRIDGE] = {}


    def load_bridge(self, model_dir: Path, filename_stem: str) -> BRIDGE | None:
        """Load (or reuse cached) BRIDGE checkpoint: `<model_dir>/<filename_stem>.pth`.

        Parameters
        ----------
        model_dir : Path
            Directory containing `.pth` checkpoints.
        filename_stem : str
            Stem used to construct checkpoint name.

        Returns
        -------
        BRIDGE | None
            Returns a loaded `BRIDGE` model in `.eval()` mode, or `None` if the file does not exist.

        Notes
        -----
        This is the main place where naming conventions matter:
        the caller uses `Path(fasta_sequence_path).stem` as `filename_stem`.
        """
        if filename_stem in self.bridge_cache:
            return self.bridge_cache[filename_stem]

        model_file = model_dir / f"{filename_stem}.pth"
        if not model_file.exists():
            logging.warning("Model not found for %s → skip", filename_stem)
            return None

        model = BRIDGE().to(self.device)
        model.load_state_dict(torch.load(model_file, map_location=self.device))
        model.eval()
        self.bridge_cache[filename_stem] = model
        return model



## 5) Run the full pipeline in-notebook

Here, we run the **entire pipeline** directly in the notebook. Instead of parsing command-line arguments, we manually set up an `argparse.Namespace` object that simulates the arguments passed via CLI.

This section demonstrates how to run the scoring process on the full dataset. You will need to:
- Provide paths to the transformer model (`Transformer_path`).
- Provide paths to save the results (`variant_out_file`).
- Run the script twice (once for `before` variation and once for `after`) to get both scores.
    

In [6]:
# ---------------------------------------------------------------------------
# Core pipeline
# ---------------------------------------------------------------------------
def process_sequences(
    names: List[str],
    seqs: List[str],
    args: argparse.Namespace,
    hub: ModelHub,
) -> None:
    """Iterate over sequences, perform modification + prediction, append results."""

    out_fp = open_output(Path(args.variant_out_file))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(2.0, device=hub.device))

    with out_fp.open("a") as fout:
        for header, seq in zip(names, seqs):
            try:
                var_pos, ref, alt, strand, seq_start = parse_variant_block(header)
            except ValueError as err:
                logging.error("%s → %s", header, err)
                continue

            # strand-aware base adjustment
            if strand == "-":
                ref, alt = apply_complement(ref), apply_complement(alt)

            # 0-based index of variant in sequence
            idx0 = var_pos - seq_start
            if idx0 < 0 or idx0 >= len(seq):
                logging.error("Variant index out of bounds (%s)", header)
                continue
            if seq[idx0] != ref:
                logging.error("Ref base mismatch (%s) — skip", header)
                continue

            if args.variation_mode == "after":
                modified_seq = substitute_base(seq, idx0, alt)
            else:
                # before variation: keep the original (reference) sequence
                modified_seq = seq

            test_emb, _ = build_Transformer_embeddings(
                sequences=[modified_seq],
                transformer_path=str(args.Transformer_path),
                device=hub.device,
                k=1,
                transpose_to_ch_first=True
            )
            test_attn = np.zeros((len(seq), 101, 103))  
            struct = np.zeros((1, 1, 101))
            motif = np.zeros((1, 1, 101))
            bio_chem = dealwithdata2(modified_seq).transpose([0, 2, 1])
            dataset = RBPInferDataset(
                embedding=test_emb, attn=test_attn, struct=struct,
                motif=motif, biochem=bio_chem
            )
            loader = DataLoader(dataset, batch_size=1, shuffle=False)

            filename_stem = Path(args.fasta_sequence_path).stem
            bridge = hub.load_bridge(Path(args.model_save_path), filename_stem)
            if bridge is None:
                continue

            prob = validate_without_sigmoid(bridge, hub.device, loader, criterion).item()
            fout.write(f"{header.lstrip('>')}\tPrediction_score:{prob:.6f}\n")
            # logging.info("Processed %s | score=%.6f", filename_stem, prob)


## 6) Tutorial-only helpers: print tensor shapes

In this section, we define utility functions for **shape-printing**. These functions are useful for debugging and tutorial purposes:

- **`show_shape`**: Prints the shape, dtype, and device of a tensor.
- **`peek_batch`**: Prints the shape of each element in a batch of data (useful for understanding the structure of data fed into the model).

These functions help users visualize tensor structures at different stages of the pipeline, ensuring everything aligns with expectations.

In [7]:
def show_shape(name, x):
    if x is None:
        print(f"{name}: None")
        return
    t = type(x).__name__
    shape = getattr(x, "shape", None)
    dtype = getattr(x, "dtype", None)
    device = getattr(x, "device", None)
    print(f"{name}: type={t} shape={shape} dtype={dtype} device={device}")

def peek_batch(loader):
    batch = next(iter(loader))
    if isinstance(batch, dict):
        for k, v in batch.items():
            show_shape(f"batch[{k}]", v)
    elif isinstance(batch, (tuple, list)):
        for i, v in enumerate(batch):
            show_shape(f"batch[{i}]", v)
    else:
        show_shape("batch", batch)



## 7) Load FASTA (uses `read_fasta()` above)

This section loads FASTA file from the dataset (`dataset_variant/AUH_HepG2.fa` in this case). The `read_fasta()` function reads the headers and sequences, which are then printed for the user to verify.

We print:
- The total number of records.
- The first few headers and a short preview of the sequence to help users confirm that the input data is loaded correctly.
    

In [8]:
# Point this to your FASTA file
FASTA_PATH = Path("dataset_variant/AUH_HepG2.fa")

print("FASTA_PATH:", FASTA_PATH)
assert FASTA_PATH.exists(), f"FASTA not found: {FASTA_PATH.resolve()}"

headers, sequences = read_fasta(FASTA_PATH)
print("Num records:", len(headers))
print("\nFirst two headers:")
for i in range(min(2, len(headers))):
    print(f"[{i+1}] {headers[i]}")
    print("    seq length:", len(sequences[i]))


FASTA_PATH: dataset_variant/AUH_HepG2.fa
Num records: 777

First two headers:
[1] >variant_1 chr1:1014401-1014501(+)[synonymous_variant]{benign} 1014451:C>T ENSG00000187608[0.001]{rs116002608}
    seq length: 101
[2] >variant_2 chr1:6599395-6599495(-)[synonymous_variant]{NA} 6599445:G>A ENSG00000162413[0.339]{rs2232460}
    seq length: 101


## 8) Single-record walkthrough + shape probes

This cell runs one FASTA record through the same logic as process_sequences() to make the pipeline transparent.

It prints:

- parsed header fields (variant_pos, REF>ALT, strand, seq_start) and the computed window index idx0
- the **before/after** sequence base at idx0
- the exact tensor shapes constructed for BRIDGE input (test_emb, test_attn, struct, motif, bio_chem) plus a DataLoader batch preview
- the expected checkpoint path: `<MODEL_SAVE_PATH>/<FASTA_STEM>.pth`
- Use this cell to quickly catch common issues (REF mismatch, out-of-bounds `idx0`, or shape inconsistencies).

In [12]:
# Mirrors per-record logic from process_sequences() for ONE record and prints shapes.

record_idx = 0
header = headers[record_idx]
seq = sequences[record_idx]

print("Selected record:", record_idx + 1)
print("Header:", header)
print("Sequence length:", len(seq))

var_pos, ref, alt, strand, seq_start = parse_variant_block(header)

print("\nParsed from header:")
print("  var_pos   :", var_pos)
print("  ref>alt   :", ref, ">", alt)
print("  strand    :", strand)
print("  seq_start :", seq_start)

if strand == "-":
    ref, alt = apply_complement(ref), apply_complement(alt)
print("Strand-adjusted ref>alt:", ref, ">", alt)

idx0 = var_pos - seq_start
print("idx0 =", idx0)
print("seq[idx0] =", seq[idx0], "| expected ref =", ref, "| match =", (seq[idx0] == ref))

variation_mode = "before"  # set "before" or "after"
if variation_mode == "after":
    modified_seq = substitute_base(seq, idx0, alt)
else:
    modified_seq = seq

print("variation_mode:", variation_mode)
print("modified_seq[idx0]:", modified_seq[idx0])

TRANSFORMER_PATH = repo_root / "RBPformer"
MODEL_SAVE_PATH = repo_root / "results/model"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    test_emb, _ = build_Transformer_embeddings(
        sequences=[modified_seq],
        transformer_path=str(TRANSFORMER_PATH),
        device=DEVICE,
        k=1,
        transpose_to_ch_first=True
    )
except Exception as e:
    print("\n[Embedding build skipped due to error]")
    print("Reason:", repr(e))
    test_emb = np.zeros((1, 768, len(modified_seq)), dtype=np.float32)

test_attn = np.zeros((len(seq), 101, 103))
struct = np.zeros((1, 1, 101))
motif = np.zeros((1, 1, 101))
bio_chem = dealwithdata2(modified_seq).transpose([0, 2, 1])

print("\nTensor shapes (as built in variant_aware.py):")
show_shape("test_emb", test_emb)
show_shape("test_attn", test_attn)
show_shape("struct", struct)
show_shape("motif", motif)
show_shape("bio_chem", bio_chem)

ds = RBPInferDataset(embedding=test_emb, attn=test_attn, struct=struct, motif=motif, biochem=bio_chem)
dl = DataLoader(ds, batch_size=1, shuffle=False)
print("\nBatch preview:")
peek_batch(dl)

expected_ckpt = MODEL_SAVE_PATH / f"{FASTA_PATH.stem}.pth"
print("\nExpected checkpoint:", expected_ckpt)
print("Exists?", expected_ckpt.exists())


Selected record: 1
Header: >variant_1 chr1:1014401-1014501(+)[synonymous_variant]{benign} 1014451:C>T ENSG00000187608[0.001]{rs116002608}
Sequence length: 101

Parsed from header:
  var_pos   : 1014451
  ref>alt   : C > T
  strand    : +
  seq_start : 1014401
Strand-adjusted ref>alt: C > T
idx0 = 50
seq[idx0] = C | expected ref = C | match = True
variation_mode: before
modified_seq[idx0]: C


Some weights of BertModel were not initialized from the model checkpoint at /fs1/private/user/wangyubo/code/BRIDGE/RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Tensor shapes (as built in variant_aware.py):
test_emb: type=ndarray shape=(1, 512, 101) dtype=float32 device=None
test_attn: type=ndarray shape=(101, 101, 103) dtype=float64 device=None
struct: type=ndarray shape=(1, 1, 101) dtype=float64 device=None
motif: type=ndarray shape=(1, 1, 101) dtype=float64 device=None
bio_chem: type=ndarray shape=(1, 99, 101) dtype=float64 device=None

Batch preview:
batch[0]: type=Tensor shape=torch.Size([1, 512, 101]) dtype=torch.float32 device=cpu
batch[1]: type=Tensor shape=torch.Size([1, 101, 103]) dtype=torch.float64 device=cpu
batch[2]: type=Tensor shape=torch.Size([1, 1, 101]) dtype=torch.float64 device=cpu
batch[3]: type=Tensor shape=torch.Size([1, 1, 101]) dtype=torch.float64 device=cpu
batch[4]: type=Tensor shape=torch.Size([1, 99, 101]) dtype=torch.float64 device=cpu

Expected checkpoint: /fs1/private/user/wangyubo/code/BRIDGE/results/model/AUH_HepG2.pth
Exists? True


## 9) CLI section

This section defines:
- **`build_argparser()`**: Sets up command-line arguments.
- **`main()`**: The script's main function that reads input FASTA files, runs the scoring pipeline, and saves results.


In [15]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def build_argparser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="after-variation scoring")
    parser.add_argument("--variation_mode", choices=["before", "after"], required=True, 
                        help="Score sequences before variation (reference) or after variation (mutated).")
    parser.add_argument("--fasta_sequence_path", required=True, type=Path)
    parser.add_argument("--variant_out_file", required=True)
    parser.add_argument("--Transformer_path", required=True, type=Path)
    parser.add_argument("--model_save_path", required=True, type=Path)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return parser


def main() -> None:
    args = build_argparser().parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    device = torch.device(args.device)

    logging.info("Running in %s_variation mode", args.variation_mode)

    logging.info("Loading FASTA from %s", args.fasta_sequence_path)
    headers, sequences = read_fasta(args.fasta_sequence_path)
    sequences_np = np.array(sequences)

    hub = ModelHub(args.Transformer_path, device)
    process_sequences(headers, sequences_np, args, hub)

    logging.info("Finished. Results appended to %s",
                 args.variant_out_file)


## 10) Compute BRIDGE scores on a small mutated (ALT) dataset and reference (REF) dataset

`after_mut` stores BRIDGE scores on the **mutated** context (used as `alt_score`).

In [16]:
!python variant_aware.py   --variation_mode after   --fasta_sequence_path ./dataset_variant/AUH_HepG2.fa   --Transformer_path ./RBPformer   --model_save_path ./results/model   --variant_out_file ./results/gwas/after_mut/AUH_HepG2_after_mut.txt   --device cuda:3

INFO: Running in after_variation mode
INFO: Loading FASTA from dataset_variant/AUH_HepG2.fa
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this mo

`before_mut` stores BRIDGE scores on the **reference** context (used as `ref_score`).

In [18]:
!python variant_aware.py   --variation_mode before   --fasta_sequence_path ./dataset_variant/AUH_HepG2.fa   --Transformer_path ./RBPformer   --model_save_path ./results/model   --variant_out_file ./results/gwas/before_mut/AUH_HepG2_before_mut.txt   --device cuda:3

INFO: Running in before_variation mode
INFO: Loading FASTA from dataset_variant/AUH_HepG2.fa
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this m

In [26]:
# Parse before/after outputs and compute delta = after - before
from pathlib import Path
print(repo_root)
before_fp = repo_root / "results/gwas/before_mut/AUH_HepG2_before_mut.txt"
after_fp  = repo_root / "results/gwas/after_mut/AUH_HepG2_after_mut.txt"

def read_pred(fp: Path):
    out = {}
    if not fp.exists():
        print("Missing:", fp)
        return out
    for line in fp.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        header, rest = line.split("\t", 1)
        score = float(rest.split("Prediction_score:")[1])
        out[header] = score
    return out

before = read_pred(before_fp)
after = read_pred(after_fp)

for k in sorted(set(before) | set(after)):
    if k in before and k in after:
        print(k)
        print("  before:", before[k])
        print("  after :", after[k])
        print("  delta :", after[k] - before[k])


/fs1/private/user/wangyubo/code/BRIDGE
variant_1 chr1:1014401-1014501(+)[synonymous_variant]{benign} 1014451:C>T ENSG00000187608[0.001]{rs116002608}
  before: -1.051359
  after : -3.360781
  delta : -2.3094219999999996
variant_10 chr1:26170778-26170878(-)[intron_variant]{NA} 26170828:C>T ENSG00000236782[0]{chr1:26497319:T:C}
  before: 0.749002
  after : 0.418629
  delta : -0.33037299999999997
variant_100 chr3:129314873-129314973(-)[3_prime_UTR_variant]{NA} 129314923:G>A ENSG00000184897[0.033]{rs8499}
  before: -5.964608
  after : -7.081443
  delta : -1.116835
variant_101 chr3:129316138-129316238(-)[5_prime_UTR_variant]{NA} 129316188:C>T ENSG00000184897[0.131]{rs14323}
  before: -2.023982
  after : -1.608301
  delta : 0.4156810000000002
variant_102 chr3:129316138-129316238(+)[intron_variant,non_coding_transcript_variant]{NA} 129316188:C>T ENSG00000206417[0.131]{rs14323}
  before: 2.414681
  after : 1.67023
  delta : -0.7444509999999998
variant_103 chr3:131381805-131381905(+)[synonymous_